Hacer dos dashboards en los que se muestren gráficos estáticos y dinámicos de diferentes tipos sobre los datos. Debe de haber gráficas de muchos tipos, también algún mapa. Las columnas son:
title: title name given to the earthquake
magnitude: The magnitude of the earthquake
date_time: date and time
cdi: The maximum reported intensity for the event range
mmi: The maximum estimated instrumental intensity for the event
alert: The alert level - “green”, “yellow”, “orange”, and “red”
tsunami: "1" for events in oceanic regions and "0" otherwise
sig: A number describing how significant the event is. Larger numbers indicate a more significant event. This value is determined on a number of factors, including: magnitude, maximum MMI, felt reports, and estimated impact
net: The ID of a data contributor. Identifies the network considered to be the preferred source of information for this event.
nst: The total number of seismic stations used to determine earthquake location.
dmin: Horizontal distance from the epicenter to the nearest station
gap: The largest azimuthal gap between azimuthally adjacent stations (in degrees). In general, the smaller this number, the more reliable is the calculated horizontal position of the earthquake. Earthquake locations in which the azimuthal gap exceeds 180 degrees typically have large location and depth uncertainties
magType: The method or algorithm used to calculate the preferred magnitude for the event
depth: The depth where the earthquake begins to rupture
latitude / longitude: coordinate system by means of which the position or location of any place on Earth's surface can be determined and described
location: location within the country
continent: continent of the earthquake hit country
country: affected country

Algunas librerías que se podrían usar son:
plotly
plotly.express
dash
panel
streamlit
geoplot
cartopy
folium
branca
matplotlib
seaborn
hvplot.pandas
mpl_toolkits.basemap
pywaffle
pypalettes
pyfonts
highlight_text
drawarrow
pandas
geopandas
mapclassify
geodatasets
numpy
netCDF4

3 dashboards con temáticas diferentes:

espacio

barplot: Número de terremotos por title, continent, country, location o alert. Visualización cambiable con dropdown menu. There is also a radio button that changes the plot between a barplot and a wordcloud
treemap: Número de terremotos por continent → country → location
dendogram: Agrupamiento jerárquico por latitude, longitude, depth, magnitude.
area chart where the y axis is latitude, being the y=0 the equator
map: Scatter geográfico estilo latitude vs longitude, tamaño por magnitude, color por alert. Los tsunamis son círculos en vez de cuadrados
cloropleth: número de terremotos, media de magnitud, mayor maginutd, media de sig (significative), número de tsunamis, interpolación entre valores de alert para conseguir uno medio por país, media de mmi.
box plots: Distribución de magnitude por continent.
also make, with an ai, a prediction of where the next earthquake will be and its magnite: Entrenar un modelo simple de regresión o un Random Forest usando: Inputs: latitude, longitude, depth, magnitude, date_time, country, continent. Outputs: next probable latitude, longitude, magnitude.

tiempo

line chart: Magnitud promedio a lo largo del tiempo (date_time). Con un radio button se convierte en lollipop plot
candlestick: Magnitud mínima, máxima, apertura y cierre por día/mes.
animation: Movimiento de epicentros en el tiempo, con tamaño por magnitud y color por alerta. Utilizar un slider donde cada "frame de la animación" es un año
Time series de alerta: Evolución de alertas por mes/año. Gráfico de líneas apiladas (stacked area chart) para ver tendencias.
Calendar heatmap: Cantidad de terremotos por día o mes (intercambiable con dropdown) del año → patrón estacional.
Polar chart (circular): ver distribución de frecuencia por hora del día o mes. Ejemplo: cantidad de terremotos por hora del día → forma circular. Alternativamente: frecuencia mensual → detectar estacionalidad visual
3d plot: plot earthquakes in an earth globe and make lines so it goes from the surface until its depth. magnitude changes the width of this line. alert determines its color. this last one can be changed by the user so the depth determines the color. Also the user can filter only tsunamis. También mostrar intensidad como una capa de calor sobre el globo (en vez de solo líneas). Color: densidad de terremotos o magnitud media. Visualmente guay si se combina con transparencia de océanos y rotación automática. Para este plot el usuario puede elegir con un rango de años los años de los terremtoso que se incluyen en el plot. Obviamente el mapa de color también cambiaría.

ninguno

violin plots: Distribución de magnitude por alert.
heatmap: correlaciones entre variables
bubble: los ejes son magnitud y profundidad, el color y el tamaño la alerta
density: distribution plot of magnitude and depth intercheangeable with a radio buttons
venn diagram: Eventos con tsunami (tsunami=1) vs alerta roja (alert="red") vs magnitud>6.
pie chart: Distribución de alertas (alert) y magtype. radio buttons. También se puede cambiar de pie chart a waffle chart con un dropdown, radio button o lo que sea
radar chart: para los top 10 terremotos de la historia. Se elige terremto con un dropdown, slider o lo que creas que es mejor

In [20]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
from scipy.stats import gaussian_kde
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
import io
import base64
from pywaffle import Waffle
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from googletrans import Translator


# https://www.kaggle.com/datasets/warcoder/earthquake-dataset/data
CARPETA_DATASETS = "dataset"
DATASET = "earthquake_1995-2023.csv"
RENDERER = "browser"  # browser o notebook
VALOR_SUSTITUTO_NULO = "desconocido"
COLOR_VALOR_SUSTITUTO_NULO = "gray"
# earthquake_1995-2023.csv tiene los datos de earthquake_data.csv más algunos adicionales

# ----------------------------
# 1. Cargar y limpiar dataset
# ----------------------------
df = pd.read_csv(os.path.join(CARPETA_DATASETS, "earthquake_1995-2023.csv"))

print(df.head(), "\n")

total_filas = df.shape[0]

nulos = df.isnull().sum()
nas = df.isna().sum()
vacios = (df == "").sum()

# Filtrar solo columnas con algún valor > 0
mask = (nulos > 0) | (nas > 0) | (vacios > 0)
conteos = pd.DataFrame({
    "Nulos": nulos[mask],
    "NA": nas[mask],
    "Vacíos": vacios[mask]
})

porcentajes = pd.DataFrame({
    "Nulos (%)": (nulos[mask] / total_filas * 100),
    "NA (%)": (nas[mask] / total_filas * 100),
    "Vacíos (%)": (vacios[mask] / total_filas * 100)
})

print(
    f"Número de filas: {total_filas}\n"
    f"Número de columnas: {df.shape[1]}\n\n"
    f"Conteos de valores por columna:\n{conteos}\n\n"
    f"Porcentaje de valores por columna:\n{porcentajes}"
)

# alert tiene 551 valores nulos/NA
# continent tiene 716 valores nulos/NA
# country tiene 349 valores nulos/NA
# location tiene 6 valores nulos/NA
for col in ["alert", "continent", "country", "location"]:
    df[col] = df[col].fillna(VALOR_SUSTITUTO_NULO)

traduccion_colores = {
    "green": "verde",
    "yellow": "amarillo",
    "red": "rojo",
    "orange": "naranja",
    VALOR_SUSTITUTO_NULO: VALOR_SUSTITUTO_NULO
}

df["alert"] = df["alert"].map(traduccion_colores)

colores_alerta = {v: k for k, v in traduccion_colores.items()}
colores_alerta[VALOR_SUSTITUTO_NULO] = COLOR_VALOR_SUSTITUTO_NULO

# traducir title?, alert, net?, magType??, location, continent, country

translator = Translator()

#for col in ["location", "country", "continent"]:
#    df[col] = df[col].astype(str).apply(lambda x: translator.translate(x, src='en', dest='es').text)

df["date_time"] = pd.to_datetime(df["date_time"], format="%d-%m-%Y %H:%M")

"""
# ----------------------------
# 2. MAPA INTERACTIVO CON FOLIUM
# ----------------------------
map_center = [df["latitude"].mean(), df["longitude"].mean()]
m = folium.Map(location=map_center, zoom_start=2, tiles="cartodb positron")

# TODO if you scroll too far horizontally, the points do not appear
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["magnitude"] ** 3 / 50,  # ** 3 / 50 o ** 2 / 10 van bien
        color="crimson" if row["alert"] == "red" else "orange" if row["alert"] == "yellow" else "blue",
        fill=True,
        fill_opacity=0.7,
        popup=f"<b>{row['title']}</b><br>Magnitude: {row['magnitude']}<br>Depth: {row['depth']} km"
    ).add_to(m)

m.save("earthquake_map.html")
print("✅ Mapa interactivo guardado en 'earthquake_map.html'")

"""

                                      title  magnitude         date_time  cdi  \
0          M 6.5 - 42 km W of Sola, Vanuatu        6.5  16-08-2023 12:47    7   
1  M 6.5 - 43 km S of Intipucá, El Salvador        6.5  19-07-2023 00:22    8   
2  M 6.6 - 25 km ESE of Loncopué, Argentina        6.6  17-07-2023 03:05    7   
3     M 7.2 - 98 km S of Sand Point, Alaska        7.2  16-07-2023 06:48    6   
4                  M 7.3 - Alaska Peninsula        7.3  16-07-2023 06:48    0   

   mmi   alert  tsunami  sig net  nst      dmin    gap magType    depth  \
0    4   green        0  657  us  114  7.177000   25.0     mww  192.955   
1    6  yellow        0  775  us   92  0.679000   40.0     mww   69.727   
2    5   green        0  899  us   70  1.634000   28.0     mww  171.371   
3    6   green        1  860  us  173  0.907000   36.0     mww   32.571   
4    5     NaN        1  820  at   79  0.879451  172.8      Mi   21.000   

   latitude  longitude               location      continent  

'\n# ----------------------------\n# 2. MAPA INTERACTIVO CON FOLIUM\n# ----------------------------\nmap_center = [df["latitude"].mean(), df["longitude"].mean()]\nm = folium.Map(location=map_center, zoom_start=2, tiles="cartodb positron")\n\n# TODO if you scroll too far horizontally, the points do not appear\nfor _, row in df.iterrows():\n    folium.CircleMarker(\n        location=[row["latitude"], row["longitude"]],\n        radius=row["magnitude"] ** 3 / 50,  # ** 3 / 50 o ** 2 / 10 van bien\n        color="crimson" if row["alert"] == "red" else "orange" if row["alert"] == "yellow" else "blue",\n        fill=True,\n        fill_opacity=0.7,\n        popup=f"<b>{row[\'title\']}</b><br>Magnitude: {row[\'magnitude\']}<br>Depth: {row[\'depth\']} km"\n    ).add_to(m)\n\nm.save("earthquake_map.html")\nprint("✅ Mapa interactivo guardado en \'earthquake_map.html\'")\n\n'

In [23]:
# Preparar datos para el radar chart (top 10 terremotos)
df_top10 = df.nlargest(10, 'magnitude').reset_index(drop=True)
df_top10['nombre_evento'] = df_top10.apply(
    lambda x: f"{x['title'][:30]}... (M{x['magnitude']})", axis=1
)

# ----------------------------
# 2. Crear App Dash
# ----------------------------
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.H1(
                "Dashboard de Terremotos - Análisis General",
                className="text-center mb-4 mt-4",
                style={'color': '#2c3e50', 'font-weight': 'bold'}
            )
        ])
    ]),
    
    # VIOLIN PLOT
    dbc.Row([
        dbc.Col([
            html.H3("Distribución de Magnitud por Alerta", className="text-center mb-3"),
            dcc.Graph(id='violin-plot', style={'height': '500px', 'width': '100%'})
        ], width=12)
    ], className="mb-5"),
    
    # HEATMAP
    dbc.Row([
        dbc.Col([
            html.H3("Mapa de Correlaciones entre Variables", className="text-center mb-3"),
            dcc.Graph(id='heatmap-correlation', style={'height': '600px', 'width': '100%'})
        ], width=12)
    ], className="mb-5"),
    
    # BUBBLE CHART
    dbc.Row([
        dbc.Col([
            html.H3("Relación Magnitud vs Profundidad (Bubble Chart)", className="text-center mb-3"),
            dcc.Graph(id='bubble-chart', style={'height': '600px', 'width': '100%'})
        ], width=12)
    ], className="mb-5"),
    
    # DENSITY PLOT
    dbc.Row([
        dbc.Col([
            html.H3("Distribución de Densidad", className="text-center mb-3"),
            html.Div([
                dbc.RadioItems(
                    id='density-radio',
                    options=[
                        {'label': ' Magnitud', 'value': 'magnitude'},
                        {'label': ' Profundidad', 'value': 'depth'}
                    ],
                    value='magnitude',
                    inline=True,
                    className="mb-3"
                )
            ], className="text-center"),
            dcc.Graph(id='density-plot', style={'height': '500px', 'width': '100%'})
        ], width=12)
    ], className="mb-5"),
    
    # VENN DIAGRAM
    dbc.Row([
        dbc.Col([
            html.H3("Diagrama de Venn: Tsunami vs Alerta Roja vs Magnitud>6",
                    className="text-center mb-3"),
            html.Img(id='venn-diagram', style={'max-width': '600px', 'width': 'auto', 'height': 'auto'})
        ], width=12, className="text-center")
    ], className="mb-5"),
    
    # PIE / WAFFLE CHART
    dbc.Row([
        dbc.Col([
            html.H3("Distribución de Alertas y Tipos de Magnitud", className="text-center mb-3"),
            html.Div([
                dbc.Row([
                    dbc.Col([
                        dbc.Label("Seleccionar variable:"),
                        dcc.RadioItems(
                            id='pie-radio',
                            options=[
                                {'label': ' Alertas', 'value': 'alert'},
                                {'label': ' Tipos de Magnitud', 'value': 'magType'}
                            ],
                            value='alert',
                            inline=True
                        )
                    ], width=6),
                    dbc.Col([
                        dbc.Label("Tipo de gráfico:"),
                        dcc.Dropdown(
                            id='chart-type-dropdown',
                            options=[
                                {'label': 'Pie Chart', 'value': 'pie'},
                                {'label': 'Waffle Chart', 'value': 'waffle'}
                            ],
                            value='pie',
                            clearable=False
                        )
                    ], width=6)
                ], className="mb-3")
            ]),
            html.Div(id='pie-waffle-container')
        ], width=12)
    ], className="mb-5"),
    
    # RADAR CHART
    dbc.Row([
        dbc.Col([
            html.H3("Radar Chart - Top 10 Terremotos", className="text-center mb-3"),
            html.Div([
                dbc.Label("Seleccionar terremoto:"),
                dcc.Dropdown(
                    id='radar-dropdown',
                    options=[{'label': name, 'value': idx} 
                             for idx, name in enumerate(df_top10['nombre_evento'])],
                    value=0,
                    clearable=False,
                    className="mb-3"
                )
            ]),
            dcc.Graph(id='radar-chart', style={'height': '600px', 'width': '100%'})
        ], width=12)
    ], className="mb-5"),
    
], fluid=True, style={'backgroundColor': '#f8f9fa'})

# ----------------------------
# 3. CALLBACKS
# ----------------------------

# VIOLIN PLOT
@app.callback(
    Output('violin-plot', 'figure'),
    Input('violin-plot', 'id')
)
def update_violin(dummy):
    fig = go.Figure()
    
    for alert_type in df['alert'].unique():
        df_filtered = df[df['alert'] == alert_type]
        color = colores_alerta.get(alert_type, 'gray')
        
        fig.add_trace(go.Violin(
            y=df_filtered['magnitude'],
            name=alert_type.capitalize(),
            box_visible=True,
            meanline_visible=True,
            fillcolor=color,
            opacity=0.6,
            line_color=color
        ))
    
    fig.update_layout(
        xaxis_title="Tipo de Alerta",
        yaxis_title="Magnitud",
        showlegend=True,
        template="plotly_white",
        height=500
    )
    
    return fig

# HEATMAP
@app.callback(
    Output('heatmap-correlation', 'figure'),
    Input('heatmap-correlation', 'id')
)
def update_heatmap(dummy):
    # Seleccionar solo columnas numéricas
    numeric_cols = ['magnitude', 'cdi', 'mmi', 'sig', 'nst', 'dmin', 'gap', 'depth', 
                    'latitude', 'longitude', 'tsunami']
    df_numeric = df[numeric_cols].dropna()
    
    corr_matrix = df_numeric.corr()
    
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale='RdBu',
        zmid=0,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        colorbar=dict(title="Correlación")
    ))
    
    fig.update_layout(
        title="Matriz de Correlación",
        xaxis_title="Variables",
        yaxis_title="Variables",
        template="plotly_white",
        height=600
    )
    
    return fig

# BUBBLE CHART
@app.callback(
    Output('bubble-chart', 'figure'),
    Input('bubble-chart', 'id')
)
def update_bubble(dummy):
    # Mapear alertas a valores numéricos para el tamaño
    alert_size_map = {
        'verde': 5,
        'amarillo': 10,
        'naranja': 15,
        'rojo': 20,
        VALOR_SUSTITUTO_NULO: 3
    }
    
    df_bubble = df.copy()
    df_bubble['bubble_size'] = df_bubble['alert'].map(alert_size_map)
    
    fig = px.scatter(
        df_bubble,
        x='magnitude',
        y='depth',
        size='bubble_size',
        color='alert',
        color_discrete_map=colores_alerta,
        hover_data=['title', 'country', 'date_time'],
        title="Magnitud vs Profundidad",
        labels={'magnitude': 'Magnitud', 'depth': 'Profundidad (km)', 'alert': 'Alerta'}
    )
    
    fig.update_layout(
        template="plotly_white",
        height=600,
        showlegend=True
    )
    
    return fig

# DENSITY PLOT
@app.callback(
    Output('density-plot', 'figure'),
    Input('density-radio', 'value')
)
def update_density(selected_var):
    data = df[selected_var].dropna()
    
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=data,
        name='Histograma',
        nbinsx=50,
        opacity=0.6,
        marker_color='steelblue'
    ))
    
    # Agregar KDE
    kde = gaussian_kde(data)
    x_range = np.linspace(data.min(), data.max(), 200)
    kde_values = kde(x_range)
    
    # Escalar KDE para que se vea bien con el histograma
    hist_counts, _ = np.histogram(data, bins=50)
    scale_factor = hist_counts.max() / kde_values.max()
    
    fig.add_trace(go.Scatter(
        x=x_range,
        y=kde_values * scale_factor,
        name='Densidad (KDE)',
        line=dict(color='red', width=3)
    ))
    
    title = f"Distribución de {'Magnitud' if selected_var == 'magnitude' else 'Profundidad'}"
    xlabel = 'Magnitud' if selected_var == 'magnitude' else 'Profundidad (km)'
    
    fig.update_layout(
        title=title,
        xaxis_title=xlabel,
        yaxis_title="Frecuencia",
        template="plotly_white",
        height=500,
        showlegend=True,
        barmode='overlay'
    )
    
    return fig

# VENN DIAGRAM
@app.callback(
    Output('venn-diagram', 'src'),
    Input('venn-diagram', 'id')
)
def update_venn(dummy):
    # Conjuntos
    set_tsunami = set(df[df['tsunami'] == 1].index)
    set_red_alert = set(df[df['alert'] == 'rojo'].index)
    set_mag_gt_6 = set(df[df['magnitude'] > 6].index)
    
    # Crear figura
    plt.figure(figsize=(10, 8))
    venn = venn3(
        [set_tsunami, set_red_alert, set_mag_gt_6],
        set_labels=('Tsunami', 'Alerta Roja', 'Magnitud > 6'),
        set_colors=('skyblue', 'lightcoral', 'lightgreen'),
        alpha=0.6
    )
    
    plt.title("Intersección de Eventos Sísmicos Significativos", fontsize=16, fontweight='bold')
    
    # Convertir a imagen base64
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', dpi=150)
    buf.seek(0)
    plt.close()
    
    img_base64 = base64.b64encode(buf.read()).decode()
    return f'data:image/png;base64,{img_base64}'

# PIE/WAFFLE CHART
@app.callback(
    Output('pie-waffle-container', 'children'),
    [Input('pie-radio', 'value'),
     Input('chart-type-dropdown', 'value')]
)
def update_pie_waffle(selected_var, chart_type):
    data_counts = df[selected_var].value_counts()
    
    if chart_type == 'pie':
        # Crear pie chart con Plotly
        colors = [colores_alerta.get(label, 'gray') if selected_var == 'alert' else None 
                  for label in data_counts.index]
        
        fig = go.Figure(data=[go.Pie(
            labels=data_counts.index,
            values=data_counts.values,
            marker=dict(colors=colors) if selected_var == 'alert' else {},
            hole=0.3
        )])
        
        title = "Distribución de Alertas" if selected_var == 'alert' else "Distribución de Tipos de Magnitud"
        fig.update_layout(
            title=title,
            template="plotly_white",
            height=500
        )
        
        return dcc.Graph(figure=fig)
    
    else:  # waffle chart
        # Crear waffle chart con matplotlib
        plt.figure(
            FigureClass=Waffle,
            rows=10,
            columns=10,
            values=data_counts.to_dict(),
            colors=[colores_alerta.get(label, 'gray') if selected_var == 'alert' else None 
                    for label in data_counts.index],
            legend={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
            figsize=(12, 8)
        )
        
        title = "Distribución de Alertas (Waffle)" if selected_var == 'alert' else "Distribución de Tipos de Magnitud (Waffle)"
        plt.title(title, fontsize=16, fontweight='bold')
        
        # Convertir a imagen base64
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', dpi=150)
        buf.seek(0)
        plt.close()
        
        img_base64 = base64.b64encode(buf.read()).decode()
        
        return html.Img(
            src=f'data:image/png;base64,{img_base64}',
            style={'width': '100%', 'max-width': '800px'}
        )

# RADAR CHART
@app.callback(
    Output('radar-chart', 'figure'),
    Input('radar-dropdown', 'value')
)
def update_radar(selected_idx):
    earthquake = df_top10.iloc[selected_idx]
    
    # Normalizar variables para el radar chart
    variables = ['magnitude', 'depth', 'sig', 'cdi', 'mmi']
    values = []
    labels = ['Magnitud', 'Profundidad', 'Significancia', 'CDI', 'MMI']
    
    for var in variables:
        val = earthquake[var]
        if pd.notna(val):
            # Normalizar a escala 0-100
            max_val = df[var].max()
            min_val = df[var].min()
            normalized = ((val - min_val) / (max_val - min_val)) * 100 if max_val != min_val else 50
            values.append(normalized)
        else:
            values.append(0)
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=labels,
        fill='toself',
        name=earthquake['nombre_evento'],
        line_color='red',
        fillcolor='rgba(255, 0, 0, 0.3)'
    ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 100]
            )
        ),
        showlegend=True,
        title=f"Características del Terremoto<br>{earthquake['title']}",
        template="plotly_white",
        height=600
    )
    
    return fig

# ----------------------------
# 4. Ejecutar App
# ----------------------------
if __name__ == '__main__':
    app.run(debug=True, port=8050)

c:\Users\jartu\AppData\Local\Programs\Python\Python312\Lib\copy.py:118: RuntimeWarning:

coroutine 'Translator.translate' was never awaited

